# CIFAR observation-process diffusion: Colab controller notebook

Use this notebook in VS Code after connecting the kernel to a Google Colab GPU runtime. The notebook assumes `cifar_observation_diffusion.py` is available in the project directory.

In [ ]:
# Check runtime and GPU.
import os, sys, platform, subprocess, textwrap
print('Python:', sys.version)
print('Platform:', platform.platform())
try:
    import torch
    print('torch:', torch.__version__)
    print('cuda available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('gpu:', torch.cuda.get_device_name(0))
except Exception as e:
    print('Torch check failed:', e)
!nvidia-smi

In [ ]:
# Mount Google Drive so code/results survive runtime restarts.
from google.colab import drive
drive.mount('/content/drive')
PROJECT_DIR = '/content/drive/MyDrive/cs229_diffusion'
os.makedirs(PROJECT_DIR, exist_ok=True)
%cd $PROJECT_DIR
!pwd
!ls -lah

In [ ]:
# Install extra dependencies. Colab usually already has CUDA-compatible torch/torchvision.
!pip -q install --upgrade scikit-learn joblib matplotlib
import torch, torchvision, sklearn, joblib, matplotlib
print('torch:', torch.__version__, 'cuda:', torch.cuda.is_available())

## Upload the script if needed

If `cifar_observation_diffusion.py` is not listed below, upload it to `MyDrive/cs229_diffusion` through Google Drive, or uncomment the upload cell.

In [ ]:
!ls -lah cifar_observation_diffusion.py || true

In [ ]:
# Optional: upload files interactively from your local machine.
# from google.colab import files
# uploaded = files.upload()
# !ls -lah

In [ ]:
# Build unsupervised 10-way pseudo-label observations.
!mkdir -p runs
!python cifar_observation_diffusion.py cluster \
  --data_dir /content/data \
  --out ./runs/clusters_lowres8_k10.joblib \
  --feature_mode lowres \
  --lowres 8 \
  --k 10 \
  --pca_dim 64 \
  --batch_size 512 \
  --num_workers 2

In [ ]:
# Smoke test: baseline isotropic model.
!python cifar_observation_diffusion.py train \
  --data_dir /content/data \
  --run_dir ./runs/base_smoke \
  --method baseline \
  --subset 5000 \
  --epochs 2 \
  --timesteps 200 \
  --base_channels 32 \
  --batch_size 64 \
  --sample_batch_size 32 \
  --num_workers 2 \
  --amp \
  --sample_every 999 \
  --save_every 1

In [ ]:
# Smoke test: estimated-discrete-latent observation model.
!python cifar_observation_diffusion.py train \
  --data_dir /content/data \
  --run_dir ./runs/cluster_smoke \
  --method cluster \
  --cluster_file ./runs/clusters_lowres8_k10.joblib \
  --subset 5000 \
  --epochs 2 \
  --timesteps 200 \
  --base_channels 32 \
  --batch_size 64 \
  --sample_batch_size 32 \
  --num_workers 2 \
  --amp \
  --sample_every 999 \
  --save_every 1

In [ ]:
# Sample from the smoke-test checkpoints.
!python cifar_observation_diffusion.py sample \
  --run_dir ./runs/base_smoke \
  --checkpoint ./runs/base_smoke/ckpt_last.pt \
  --method baseline \
  --cluster_file ./runs/clusters_lowres8_k10.joblib \
  --n 64 \
  --sample_batch_size 32 \
  --use_ema \
  --out samples_base_smoke.png

!python cifar_observation_diffusion.py sample \
  --run_dir ./runs/cluster_smoke \
  --checkpoint ./runs/cluster_smoke/ckpt_last.pt \
  --method cluster \
  --cluster_file ./runs/clusters_lowres8_k10.joblib \
  --n 64 \
  --sample_batch_size 32 \
  --use_ema \
  --out samples_cluster_smoke.png

In [ ]:
# Optional: first real run, smaller than final. Run baseline and cluster separately.
# !python cifar_observation_diffusion.py train \
#   --data_dir /content/data \
#   --run_dir ./runs/base_colab_20ep \
#   --method baseline \
#   --epochs 20 \
#   --timesteps 500 \
#   --base_channels 64 \
#   --batch_size 128 \
#   --sample_batch_size 64 \
#   --num_workers 2 \
#   --amp \
#   --sample_every 999 \
#   --save_every 5